In [1]:
from collections import Counter
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


data = pd.read_excel(r"E:\desktop\Multiobjective Identification of Phosphor Materials via a Domain Knowledge Integrated Semisupervised Learning Framework\dataset\stage2\525×87.xlsx")
X = data.iloc[:, 3:]
X
feature_names = X.columns
X = StandardScaler().fit_transform(X)

min_val = np.min(X, axis=0)
max_val = np.max(X, axis=0)
X = (X - min_val) / (max_val - min_val)
imputer = SimpleImputer(strategy="mean")
X = imputer.fit_transform(X)
X.shape

(525, 87)

In [2]:

def extract_numbers(file_path):
    df = pd.read_excel(file_path)
    valid_numbers = []
    for index, value in df['composition'].items():
        if not pd.isna(value):
            valid_numbers.append(int(df.at[index, 'number']))
    return valid_numbers

file_path = r"E:\desktop\Multiobjective Identification of Phosphor Materials via a Domain Knowledge Integrated Semisupervised Learning Framework\dataset\stage2\benchmark2.xlsx"
reported_structures = extract_numbers(file_path)
print(reported_structures)


def remove_noise_by_nearest_cluster(X, labels):
    """"""

    noise_mask = labels == -1


    if not any(noise_mask):
        return labels


    core_samples = ~noise_mask
    X_core = X[core_samples]
    y_core = labels[core_samples]


    if len(X_core) == 0:

        new_labels = np.zeros_like(labels)
        return new_labels


    nn = NearestNeighbors(n_neighbors=1).fit(X_core)
    distances, indices = nn.kneighbors(X[noise_mask])


    new_labels = labels.copy()
    new_labels[noise_mask] = y_core[indices.flatten()]
    return new_labels

for j in range(30, 100, 5):
    j = j/100

    pca = PCA(n_components= j)
    pca_data = pca.fit_transform(X)
    num_pca = pca_data.shape[1]

    for eps in range(1,100,1):
        eps = eps/100
        for min_samples in range(6, 21,3):
            db = DBSCAN(eps=eps, metric='euclidean', min_samples=min_samples)
            db.fit(pca_data)
            labels = db.labels_
            labels = remove_noise_by_nearest_cluster(pca_data, labels)
            if 7 > len(np.unique(labels)) > 1:
                print(f"\nSilhouette score: {silhouette_score(pca_data, labels):.2f}", f"\nCalinski-Harabasz score: {calinski_harabasz_score(pca_data, labels):.2f}")
                data['cluster_label'] = labels
                unique_labels = np.unique(labels)
                print( f"\nPCA={j}",f"\neps={eps}", f"\nmin_samples={min_samples}", f"\nNumber of clusters={len(unique_labels)}")

                dict = {}

                label_1 = []
                for num  in  reported_structures:
                        label = labels[num-1]
                        label_1.append(int(label))
                print(label_1)

                for label in unique_labels:

                    cluster_samples = data[data['cluster_label'] == label]
                    total_in_cluster = len(cluster_samples)


                    reported_in_cluster = len([num for num in cluster_samples['number']
                                            if num in reported_structures])

                    print(f"Cluster {label}: {reported_in_cluster}/{total_in_cluster} "
                        f"(Reported ratio: {reported_in_cluster/total_in_cluster:.2%})")


                # print("Label distribution:", Counter(labels))

            else:
                    pass



[18, 19, 20, 21, 175, 206, 273, 276, 287, 291, 309, 320, 358, 471, 507]

Silhouette score: 0.41 
Calinski-Harabasz score: 422.29

PCA=0.3 
eps=0.04 
min_samples=6 
Number of clusters=3
[2, 2, 2, 1, 2, 0, 1, 0, 1, 2, 2, 1, 2, 2, 2]
Cluster 0: 2/90 (Reported ratio: 2.22%)
Cluster 1: 4/127 (Reported ratio: 3.15%)
Cluster 2: 9/308 (Reported ratio: 2.92%)

Silhouette score: 0.25 
Calinski-Harabasz score: 224.50

PCA=0.3 
eps=0.06 
min_samples=9 
Number of clusters=5
[4, 4, 4, 2, 4, 2, 3, 2, 0, 4, 4, 0, 4, 4, 4]
Cluster 0: 2/32 (Reported ratio: 6.25%)
Cluster 1: 0/50 (Reported ratio: 0.00%)
Cluster 2: 3/123 (Reported ratio: 2.44%)
Cluster 3: 1/51 (Reported ratio: 1.96%)
Cluster 4: 9/269 (Reported ratio: 3.35%)

Silhouette score: 0.34 
Calinski-Harabasz score: 342.10

PCA=0.3 
eps=0.08 
min_samples=12 
Number of clusters=3
[2, 2, 2, 0, 2, 0, 1, 0, 0, 2, 2, 0, 2, 2, 2]
Cluster 0: 5/195 (Reported ratio: 2.56%)
Cluster 1: 1/65 (Reported ratio: 1.54%)
Cluster 2: 9/265 (Reported ratio: 3.40%)

Sil

In [3]:
pca = PCA(n_components= 0.5)
pca_data = pca.fit_transform(X)
print(pca_data.shape)
ratios = pca.explained_variance_ratio_
components = pca.components_
X = pd.DataFrame(X)

components_df = pd.DataFrame(
    components,
    columns=X.columns,
    index=[f"PC{i+1}" for i in range(components.shape[0])]
)

output_file = "pca_components.xlsx"
components_df.to_excel(output_file, index=True)
db = DBSCAN(eps=0.37,metric='euclidean', min_samples=9)
db.fit(pca_data)
labels = db.labels_
labels = remove_noise_by_nearest_cluster(pca_data, labels)

df = pd.DataFrame(labels)
df.to_excel("DBSCAN_result_1.xlsx", index=False)

print(f"\nSilhouette score: {silhouette_score(pca_data, labels):.2f}", f"\nCalinski-Harabasz score: {calinski_harabasz_score(pca_data, labels):.2f}")


data['cluster_label'] = labels
unique_labels = np.unique(labels)

dict = {}

label_1 = []
for num  in  reported_structures:
    label = labels[num-1]
    label_1.append(int(label))
print(label_1)

for label in unique_labels:

    cluster_samples = data[data['cluster_label'] == label]
    total_in_cluster = len(cluster_samples)


    reported_in_cluster = len([num for num in cluster_samples['number']
                                        if num in reported_structures])

    print(f"Cluster {label}: {reported_in_cluster}/{total_in_cluster} "
                    f"(Reported ratio: {reported_in_cluster/total_in_cluster:.2%})")


print("Label distribution:", Counter(labels))

(525, 5)

Silhouette score: 0.15 
Calinski-Harabasz score: 130.49
[3, 3, 3, 0, 3, 0, 4, 0, 0, 4, 3, 0, 3, 3, 3]
Cluster 0: 5/147 (Reported ratio: 3.40%)
Cluster 1: 0/101 (Reported ratio: 0.00%)
Cluster 2: 0/50 (Reported ratio: 0.00%)
Cluster 3: 8/156 (Reported ratio: 5.13%)
Cluster 4: 2/71 (Reported ratio: 2.82%)
Label distribution: Counter({np.int64(3): 156, np.int64(0): 147, np.int64(1): 101, np.int64(4): 71, np.int64(2): 50})


In [4]:
import pandas as pd
from sklearn.decomposition import PCA
from openpyxl.styles import Font
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl import Workbook


wb = Workbook()
ws = wb.active


for r in dataframe_to_rows(components_df, index=True, header=True):
    ws.append(r)


red_bold_font = Font(color="FF0000", bold=True)


for row in range(2, ws.max_row + 1):

    values = [ws.cell(row=row, column=col).value for col in range(2, ws.max_column + 1)]


    values = [0 if v is None else v for v in values]


    abs_values = [abs(v) for v in values]
    top_10_indices = sorted(range(len(abs_values)), key=lambda i: abs_values[i], reverse=True)[:10]


    for idx in top_10_indices:
        ws.cell(row=row, column=idx + 2).font = red_bold_font


output_file = "pca_components_highlighted.xlsx"
wb.save(output_file)

print(f"The component-loading matrix has been saved to {output_file}, with the 10 largest absolute values highlighted.")

The component-loading matrix has been saved to pca_components_highlighted.xlsx, with the 10 largest absolute values highlighted.
